# P5 — Sequence Modeling with RNNs

**Goal:** Apply basic and advanced **RNN / LSTM / GRU, Encoder-Decoder** architectures to a sequence-based sentiment classification task.

---

## Dataset (Required)

### IMDB Movie Reviews (Binary Sentiment Classification)
**Access method (required):**
```python
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = ...
```

You must create a validation split from training data and pad/truncate sequences to a fixed length.

---
## What you will implement

You will implement and compare the following **sequence models**:

- **Vanilla RNN**
- **LSTM**
- **GRU**
- **Stacked LSTM + Dropout**
- **Encoder–Decoder LSTM**
- **LSTM + GRU Hybrid**

---

## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [1]:
# ============================================================
# Q0) Environment Setup
# ============================================================

import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


TensorFlow: 2.20.0


---
## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q11)** and continues with the **Markdown responses (Q12–Q15)**.  
These correspond to the questions listed in the assignment description on **Canvas**. Follow the instructions provided in the **preceding Markdown cells** for each step.

### Tasks

This assignment focuses on **sequence modeling for text classification** using recurrent neural networks.

You will:

- Train and evaluate the following **sequence models**:
  - **Vanilla RNN**
  - **LSTM**
  - **GRU**

- Implement additional **advanced architectures**:
  - **Stacked LSTM + Dropout**
  - **Encoder–Decoder LSTM**
  - **LSTM + GRU Hybrid**

- Use the **IMDB Movie Reviews dataset** for **binary sentiment classification**.

- Perform a **comparative analysis** of the models, including:
  - training convergence behavior
  - validation and test performance
  - explanation of architectural differences across models

Ensure that all models are **computationally feasible** to train on **CPU-only environments** by using the recommended hyperparameters unless you have access to a GPU (e.g. Google Colab).

---

## Q1 — Load Dataset & Inspect

Use the **IMDB Movie Reviews dataset** from Keras and inspect its basic structure.

### Student Tasks

- Load the IMDB dataset using `tensorflow.keras.datasets.imdb` with a vocabulary size of **10,000 words** by frequency.

- Split the training data into **training** and **validation** sets.

- Inspect the dataset by:
  - Printing the **number of training and test samples**
  - Displaying the **label distribution (train)**
  - Printing one example **Sequence length (train)**

---

In [2]:
# ============================================================
# Question Q1 — Load IMDB Dataset (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Load the IMDB Movie Reviews dataset
# 2) Print the number of training and test examples
# 3) Check the label distribution in the training set
# 4) Inspect sequence length statistics
# ============================================================

import numpy as np
from tensorflow.keras.datasets import imdb

# TODO 1: Define vocabulary size (keep the top 10,000 most frequent words)
VOCAB_SIZE = 10000

# TODO 2: Load the IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# TODO 3: Print dataset sizes
print("Train examples:", len(x_train))
print("Test examples: ", len(x_test))

# TODO 4: Print label distribution (should be balanced 50/50 for IMDB)
print("Label distribution (train):", np.unique(y_train, return_counts=True))

# TODO 5: Compute sequence length statistics
train_lengths = np.array([len(s) for s in x_train])

print(
    "Sequence length (train): min/median/max =",
    train_lengths.min(),
    int(np.median(train_lengths)),
    train_lengths.max()
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train examples: 25000
Test examples:  25000
Label distribution (train): (array([0, 1]), array([12500, 12500]))
Sequence length (train): min/median/max = 11 178 2494


---

## Q2 — Validation Split & Sequence Padding

Prepare the dataset for training by creating a **validation split** and converting all sequences to a **fixed length**.

### Student Tasks

- Create a **validation set** from the training data (`VAL_SIZE = 5000`).  
  Use a **deterministic split from the end of the training set**.

- Define a maximum sequence length **`MAX_LEN`** (e.g., 200–300 tokens) based on the sequence statistics observed in **Q1**.

- Apply **sequence padding and truncation** using `pad_sequences` so that all reviews have the same length:
  - Use **post-padding**
  - Use **post-truncation**

- Generate padded datasets for:
  - `x_train_pad`
  - `x_val_pad`
  - `x_test_pad`

- Keep it **consistent across all models**.

- Print the shapes of the padded arrays to confirm the preprocessing step completed successfully.

---

In [3]:
# ============================================================
# Question Q2 — Validation Split & Sequence Padding (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define MAX_LEN and validation size
# 2) Create a validation split from the training data
# 3) Pad/truncate sequences to a fixed length
# 4) Verify resulting dataset shapes
# ============================================================

from tensorflow.keras.preprocessing.sequence import pad_sequences

# TODO 1: Define sequence length and validation size
MAX_LEN = 250    # Standard length capturing the median/bulk of IMDB reviews
VAL_SIZE = 5000  # 20% of the 25k training set

# TODO 2: Create validation split from the end of the training set
x_val, y_val = x_train[-VAL_SIZE:], y_train[-VAL_SIZE:]
x_train2, y_train2 = x_train[:-VAL_SIZE], y_train[:-VAL_SIZE]

# TODO 3: Apply padding and truncation
# Post-padding/truncating ensures the RNN reads the start of the review first and pads the end.
x_train_pad = pad_sequences(
    x_train2,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_val_pad = pad_sequences(
    x_val,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

# Print dataset shapes
print("x_train_pad:", x_train_pad.shape)
print("x_val_pad:  ", x_val_pad.shape)
print("x_test_pad: ", x_test_pad.shape)

x_train_pad: (20000, 250)
x_val_pad:   (5000, 250)
x_test_pad:  (25000, 250)


---

## Q3 — Build `tf.data` Pipelines

Create efficient **data pipelines** for training, validation, and testing using **TensorFlow `tf.data`**.

### Student Tasks

- Convert the padded datasets into **TensorFlow datasets** using `tf.data.Dataset.from_tensor_slices`.

- Create datasets for:
  - **training**
  - **validation**
  - **testing**

- Apply the following pipeline steps:
  - **shuffle** the training dataset
  - **batch** the datasets using an appropriate batch size
  - use **prefetching** to improve training performance

- Ensure the pipelines are ready to be used directly in **model training with `model.fit()`**.


---

In [4]:
# ============================================================
# Question Q3 — tf.data Pipelines (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define batch size and AUTOTUNE
# 2) Create TensorFlow datasets from the padded arrays
# 3) Apply shuffle, batch, and prefetch operations
# 4) Prepare pipelines for training, validation, and testing
# ============================================================

# TODO 1: Define batch size and autotune
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# TODO 2: Create training dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train_pad, y_train2))

# TODO 3: Apply shuffle, batch, and prefetch
train_ds = train_ds.shuffle(10000, seed=42).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 4: Create validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((x_val_pad, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 5: Create test dataset
test_ds = tf.data.Dataset.from_tensor_slices((x_test_pad, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Pipelines ready.")

Pipelines ready.


---

## Q4 — Define Training Utilities

In this step, you will create **reusable helper functions** that simplify model training, evaluation, and experiment management.

These utilities will be used for **all models later in the assignment**.

### Student Tasks

1. **Create training callbacks**

   Define a function that returns commonly used training callbacks, including:

   - **EarlyStopping** to stop training when validation performance stops improving.
   - **ReduceLROnPlateau** to automatically reduce the learning rate when validation loss plateaus.
   - **ModelCheckpoint** to save the **best-performing model** during training.

2. **Define a model compilation function**

   Implement a function that compiles a model using:

   - **Adam optimizer**
   - **Binary cross-entropy loss** for sentiment classification
   - **Accuracy** as the evaluation metric

3. **Create a training function**

   Implement a function that trains a model using:

   - the **training dataset**
   - the **validation dataset**
   - the callbacks defined above

4. **Create an evaluation function**

   Implement a function that evaluates a trained model on a dataset and reports:

   - **loss**
   - **accuracy**

These functions will help keep the notebook **organized, reusable, and consistent across experiments**.

---

In [5]:
# ============================================================
# Question Q4 — Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define callbacks for training
# 2) Compile the model with optimizer, loss, and metrics
# 3) Train the model using training and validation datasets
# 4) Evaluate the trained model on a dataset
# ============================================================

# TODO 1: Define training callbacks
def build_callbacks(run_name: str):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{run_name}.keras",
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]

# TODO 2: Compile model
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=["accuracy"],
    )
    return model

# TODO 3: Train model
def train_model(model, run_name: str, epochs=8):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=build_callbacks(run_name),
        verbose=1,
    )
    return history

# TODO 4: Evaluate model
def evaluate_model(model, name: str, ds):
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name}: loss={loss:.4f}, acc={acc:.4f}")
    return loss, acc

---

## Q5 — Model A: Vanilla RNN

Build a **Vanilla RNN** model for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add a **SimpleRNN layer** for sequence processing.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.

---

In [6]:
# ============================================================
# Question Q5 — Model A: Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define embedding dimension and RNN units
# 2) Build a Sequential RNN model
# 3) Add Input, Embedding, RNN, and Dense layers
# 4) Compile the model
# 5) Display the model summary
# ============================================================

from tensorflow.keras import layers

# TODO 1: Define model hyperparameters
EMBED_DIM = 64
RNN_UNITS = 32

# TODO 2: Build the Sequential model
rnn_model = tf.keras.Sequential([

    # TODO 3: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 4: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 5: Simple RNN layer
    layers.SimpleRNN(RNN_UNITS),

    # TODO 6: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="vanilla_rnn")

# TODO 7: Compile the model
rnn_model = compile_model(rnn_model, lr=1e-3)

# Print model summary
rnn_model.summary()

# ------------------------------------------------------------
# Train and Evaluate
# ------------------------------------------------------------
# TODO 1: Train the RNN model
history_rnn = train_model(rnn_model, run_name="proj5_vanilla_rnn", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(rnn_model, "Validation (RNN)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(rnn_model, "Test (RNN)", test_ds)

Model: "vanilla_rnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 643,137 (2.45 MB)

 Trainable params: 643,137 (2.45 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - accuracy: 0.5046 - loss: 0.6950 - val_accuracy: 0.5114 - val_loss: 0.6939 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5990 - loss: 0.6511 - val_accuracy: 0.5018 - val_loss: 0.7313 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.6750 - loss: 0.5132 - val_accuracy: 0.4912 - val_loss: 0.7789 - learning_rate: 5.0000e-04
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.7234 - loss: 0.4519 - val_accuracy: 0.4950 - val_loss: 0.8172 - learning_rate: 2.5000e-04
Validation (RNN): loss=0.6939, acc=0.5114
Test (RNN): loss=0.6938, acc=0.5028


(0.6937718987464905, 0.5027999877929688)

In [7]:
# ============================================================
# Train and Evaluate Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the Vanilla RNN model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Build the Sequential LSTM model
lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm")

# TODO 6: Compile the model
lstm_model = compile_model(lstm_model, lr=1e-3)

# Print model summary
lstm_model.summary()

# ------------------------------------------------------------
# Train and Evaluate
# ------------------------------------------------------------
# TODO 1: Train the LSTM model
history_lstm = train_model(lstm_model, run_name="proj5_lstm", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(lstm_model, "Validation (LSTM)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(lstm_model, "Test (LSTM)", test_ds)

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 652,449 (2.49 MB)

 Trainable params: 652,449 (2.49 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.5081 - loss: 0.6928 - val_accuracy: 0.5324 - val_loss: 0.6915 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.5763 - loss: 0.6557 - val_accuracy: 0.5996 - val_loss: 0.6366 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.6131 - loss: 0.5998 - val_accuracy: 0.5730 - val_loss: 0.6446 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.6471 - loss: 0.5433 - val_accuracy: 0.8146 - val_loss: 0.5572 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.8543 - loss: 0.3930 - val_accuracy: 0.8186 - val_loss: 0.4630 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8875 - loss: 0.3029 - val_accuracy: 0.8328 - val_loss: 0.5133 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9003 - lo

(0.4536571800708771, 0.8289999961853027)

---

## Q6 — Model B: LSTM

Build an **LSTM-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add an **LSTM layer** to capture long-term dependencies in sequences.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [8]:
# ============================================================
# Question Q6 — Model B: LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build an LSTM-based sequence model
# 2) Add Input, Embedding, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential LSTM model
lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm")

# TODO 6: Compile the model
lstm_model = compile_model(lstm_model, lr=1e-3)

# Print model summary
lstm_model.summary()

# ------------------------------------------------------------
# Train and Evaluate
# ------------------------------------------------------------
# TODO 1: Train the LSTM model
history_lstm = train_model(lstm_model, run_name="proj5_lstm", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(lstm_model, "Validation (LSTM)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(lstm_model, "Test (LSTM)", test_ds)

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 652,449 (2.49 MB)

 Trainable params: 652,449 (2.49 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.5300 - loss: 0.6914 - val_accuracy: 0.5666 - val_loss: 0.6778 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.5590 - loss: 0.6836 - val_accuracy: 0.5452 - val_loss: 0.6813 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5503 - loss: 0.6688 - val_accuracy: 0.5538 - val_loss: 0.6763 - learning_rate: 5.0000e-04
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.5963 - loss: 0.6318 - val_accuracy: 0.5950 - val_loss: 0.6405 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6229 - loss: 0.5938 - val_accuracy: 0.6082 - val_loss: 0.6293 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.7911 - loss: 0.4635 - val_accuracy: 0.8066 - val_loss: 0.4873 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8281 -

(0.5117242932319641, 0.798039972782135)

In [9]:
# ============================================================
# Train and Evaluate LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the LSTM model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================
# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=1e-3)

# Print model summary
gru_model.summary()

# ------------------------------------------------------------
# Train and Evaluate
# ------------------------------------------------------------
# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 649,441 (2.48 MB)

 Trainable params: 649,441 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.5070 - loss: 0.6930 - val_accuracy: 0.5114 - val_loss: 0.6915 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.5748 - loss: 0.6630 - val_accuracy: 0.5734 - val_loss: 0.6618 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6465 - loss: 0.5786 - val_accuracy: 0.8182 - val_loss: 0.4450 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8796 - loss: 0.2995 - val_accuracy: 0.8676 - val_loss: 0.3321 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9318 - loss: 0.1879 - val_accuracy: 0.8656 - val_loss: 0.3371 - learning_rate: 0.0010
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9662 - loss: 0.1103 - val_accuracy: 0.8758 - val_loss: 0.3595 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9802 - loss: 0.075

(0.39551711082458496, 0.8556399941444397)

---

## Q7 — Model C: GRU

Build a **GRU-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add **Embedding → GRU → Dense(sigmoid)** layers.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [10]:
# ============================================================
# Question Q7 — Model C: GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a GRU-based sequence model
# 2) Add Input, Embedding, GRU, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=1e-3)

# Print model summary
gru_model.summary()

# ------------------------------------------------------------
# Train and Evaluate
# ------------------------------------------------------------
# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 649,441 (2.48 MB)

 Trainable params: 649,441 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.5041 - loss: 0.6933 - val_accuracy: 0.5178 - val_loss: 0.6924 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5641 - loss: 0.6807 - val_accuracy: 0.6152 - val_loss: 0.6641 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8000 - loss: 0.4578 - val_accuracy: 0.8190 - val_loss: 0.4282 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8894 - loss: 0.2829 - val_accuracy: 0.8464 - val_loss: 0.3728 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9341 - loss: 0.1896 - val_accuracy: 0.8594 - val_loss: 0.3707 - learning_rate: 0.0010
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9572 - loss: 0.1311 - val_accuracy: 0.8574 - val_loss: 0.4153 - learning_rate: 0.0010
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9785 - loss: 0.0806 - 

(0.39612889289855957, 0.8483200073242188)

In [11]:
# ============================================================
# Train and Evaluate GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the GRU model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=1e-3)

# Print model summary
gru_model.summary()

# ------------------------------------------------------------
# Train and Evaluate
# ------------------------------------------------------------
# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 649,441 (2.48 MB)

 Trainable params: 649,441 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.5063 - loss: 0.6931 - val_accuracy: 0.5128 - val_loss: 0.6921 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.5641 - loss: 0.6704 - val_accuracy: 0.5596 - val_loss: 0.6711 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6166 - loss: 0.5942 - val_accuracy: 0.6166 - val_loss: 0.7139 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6982 - loss: 0.5315 - val_accuracy: 0.5376 - val_loss: 0.6872 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6542 - loss: 0.5295 - val_accuracy: 0.5952 - val_loss: 0.6668 - learning_rate: 2.5000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.7067 - loss: 0.4887 - val_accuracy: 0.7382 - val_loss: 0.6037 - learning_rate: 2.5000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8411 - los

(0.559228777885437, 0.790120005607605)

---

## 8) Complex Model D — Stacked LSTM with Dropout

In this question, build a **deeper LSTM-based sequence classifier** using two recurrent layers.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `Dropout(...)`
  - `LSTM(...)`
  - `Dense(1, activation="sigmoid")`
- Train the model using the same optimizer and callbacks.
- Evaluate on validation and test sets.
- Compare it with the single-layer LSTM from Q6.

### Goal
Study whether **stacking recurrent layers** helps the model learn richer sequential sentiment patterns.


---

In [12]:
# ============================================================
# Question Q8 — Complex Model D: Stacked LSTM + Dropout (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a stacked LSTM model with dropout
# 2) Add Input, Embedding, LSTM, Dropout, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential stacked LSTM model
stacked_lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: First LSTM layer (must return sequences to feed into the next LSTM)
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: Dropout layer
    layers.Dropout(0.5),

    # TODO 6: Second LSTM layer
    layers.LSTM(RNN_UNITS // 2),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="stacked_lstm_dropout")

# TODO 8: Compile the model
stacked_lstm_model = compile_model(stacked_lstm_model, lr=1e-3)

# Print model summary
stacked_lstm_model.summary()

# TODO 9: Train the model
history_stacked_lstm = train_model(
    stacked_lstm_model,
    run_name="proj5_stacked_lstm_dropout",
    epochs=8
)

# TODO 10: Evaluate on validation set
evaluate_model(stacked_lstm_model, "Validation (Stacked LSTM + Dropout)", val_ds)

# TODO 11: Evaluate on test set
evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)

Model: "stacked_lstm_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 250, 32)        │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 250, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 655,569 (2.50 MB)

 Trainable params: 655,569 (2.50 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.5157 - loss: 0.6931 - val_accuracy: 0.5062 - val_loss: 0.6938 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.5199 - loss: 0.6898 - val_accuracy: 0.5458 - val_loss: 0.6772 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.5899 - loss: 0.6563 - val_accuracy: 0.6002 - val_loss: 0.6558 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.6112 - loss: 0.6543 - val_accuracy: 0.6526 - val_loss: 0.6531 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.6620 - loss: 0.6249 - val_accuracy: 0.7516 - val_loss: 0.5638 - learning_rate: 0.0010
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7513 - loss: 0.5545 - val_accuracy: 0.7216 - val_loss: 0.6290 - learning_rate: 0.0010
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.7873 - loss: 0.5034 

(0.5297449827194214, 0.7649999856948853)

---

## 9) Complex Model E — Encoder–Decoder LSTM Classifier

In this question, build an **encoder–decoder style recurrent model** for sentiment classification.

### Idea
- The **encoder LSTM** reads the review and produces a compact context representation.
- A `RepeatVector` creates a short decoded sequence from that context.
- A **decoder LSTM** transforms the context into a richer hidden representation.
- A final dense layer predicts the review sentiment.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM` encoder
  - `RepeatVector(1)`
  - `LSTM` decoder
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it with the simpler one-layer LSTM.

### Goal
Explore whether a **more structured encoder–decoder design** is useful for sequence classification, even though it is more common in seq2seq tasks.


---

In [13]:
# ============================================================
# Question Q9 — Complex Model E: Encoder-Decoder LSTM Classifier (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define the input layer
# 2) Add Embedding, Encoder LSTM, RepeatVector, Decoder LSTM, and Dense layers
# 3) Build the functional model
# 4) Compile the model using the training utility
# 5) Train the model
# 6) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Define input layer
encdec_inputs = layers.Input(shape=(MAX_LEN,))

# TODO 2: Embedding layer
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(encdec_inputs)

# TODO 3: Encoder LSTM
encoded = layers.LSTM(RNN_UNITS)(x)

# TODO 4: Repeat encoded representation
decoded = layers.RepeatVector(1)(encoded)

# TODO 5: Decoder LSTM
decoded = layers.LSTM(RNN_UNITS // 2)(decoded)

# TODO 6: Output layer
encdec_outputs = layers.Dense(1, activation="sigmoid")(decoded)

# TODO 7: Build functional model
encdec_model = tf.keras.Model(inputs=encdec_inputs, outputs=encdec_outputs, name="encdec_lstm_classifier")

# TODO 8: Compile the model
encdec_model = compile_model(encdec_model, lr=1e-3)

# TODO 9: Print model summary
encdec_model.summary()

# TODO 10: Train the model
history_encdec = train_model(
    encdec_model,
    run_name="proj5_encdec_lstm",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(encdec_model, "Validation (Encoder-Decoder LSTM)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)

Model: "encdec_lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 250)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_7 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 655,569 (2.50 MB)

 Trainable params: 655,569 (2.50 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5134 - loss: 0.6913 - val_accuracy: 0.5160 - val_loss: 0.6920 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.5986 - loss: 0.6408 - val_accuracy: 0.7778 - val_loss: 0.5271 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8347 - loss: 0.3984 - val_accuracy: 0.8562 - val_loss: 0.3439 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9086 - loss: 0.2488 - val_accuracy: 0.8596 - val_loss: 0.3487 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9514 - loss: 0.1516 - val_accuracy: 0.8636 - val_loss: 0.3803 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9694 - loss: 0.1063 - val_accuracy: 0.8672 - val_loss: 0.4072 - learning_rate: 2.5000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9779 - loss: 0

(0.4348560869693756, 0.8548799753189087)

---

## 10) Complex Model F — LSTM + GRU Hybrid

In this question, build a **hybrid recurrent architecture** that combines LSTM and GRU layers without using bidirectional processing.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `GRU(...)`
  - `Dropout`
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it against all earlier models in terms of accuracy and complexity.

### Goal
Test whether combining **LSTM-based memory** with a **GRU-based final sequence encoder** captures richer sentiment patterns than a single recurrent layer.


---

In [14]:
# ============================================================
# Question Q10 — Complex Model F: LSTM + GRU Hybrid (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a hybrid sequence model using LSTM and GRU layers
# 2) Add Input, Embedding, LSTM, GRU, Dropout, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential hybrid model
hybrid_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: GRU layer
    layers.GRU(RNN_UNITS // 2, dropout=0.2, recurrent_dropout=0.0),

    # TODO 6: Dropout layer
    layers.Dropout(0.5),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm_gru_hybrid")

# TODO 8: Compile the model
hybrid_model = compile_model(hybrid_model, lr=1e-3)

# TODO 9: Print model summary
hybrid_model.summary()

# TODO 10: Train the model
history_hybrid = train_model(
    hybrid_model,
    run_name="proj5_lstm_gru_hybrid",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(hybrid_model, "Validation (LSTM + GRU Hybrid)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

Model: "lstm_gru_hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 250, 32)        │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 16)             │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 654,833 (2.50 MB)

 Trainable params: 654,833 (2.50 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.5045 - loss: 0.6932 - val_accuracy: 0.5314 - val_loss: 0.6904 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.5605 - loss: 0.6717 - val_accuracy: 0.5830 - val_loss: 0.6548 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6241 - loss: 0.5968 - val_accuracy: 0.6026 - val_loss: 0.6363 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6398 - loss: 0.5525 - val_accuracy: 0.5296 - val_loss: 0.6812 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.7795 - loss: 0.4520 - val_accuracy: 0.8332 - val_loss: 0.4447 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.8939 - loss: 0.2932 - val_accuracy: 0.8536 - val_loss: 0.4054 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.9194 - loss: 

(0.4648089110851288, 0.8434399962425232)

---

## Q11 — Performance Comparison Table

Create a compact table comparing all completed models:

- Vanilla RNN
- LSTM
- GRU
- Stacked LSTM + Dropout
- Encoder–Decoder LSTM
- LSTM + GRU Hybrid

### Student Tasks
- Create a comparison table summarizing the results of all models.
- Include validation accuracy and test accuracy.
- Identify the best-performing recurrent model.
- Briefly comment on whether more complex architectures were helpful.


---

In [15]:
# ============================================================
# Question Q11 — Performance Comparison Table (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define helper functions for validation and training accuracy
# 2) Evaluate all trained models on the test dataset
# 3) Store model names and metrics in summary rows
# 4) Create a pandas DataFrame for comparison
# 5) Sort the table by test accuracy
# ============================================================

import pandas as pd

# TODO 1: Best validation accuracy from training history
def best_val_acc(history):
    return np.max(history.history.get("val_accuracy", [np.nan]))

# TODO 2: Final training accuracy from training history
def final_train_acc(history):
    return history.history.get("accuracy", [np.nan])[-1]

summary_rows = []

# TODO 3: Evaluate all models on the test set
_, rnn_test_acc = evaluate_model(rnn_model, "Test (Vanilla RNN)", test_ds)
_, lstm_test_acc = evaluate_model(lstm_model, "Test (LSTM)", test_ds)
_, gru_test_acc = evaluate_model(gru_model, "Test (GRU)", test_ds)
_, stacked_lstm_test_acc = evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)
_, encdec_test_acc = evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)
_, hybrid_test_acc = evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

# TODO 4: Append summary rows
summary_rows.append(["Vanilla RNN", final_train_acc(history_rnn), best_val_acc(history_rnn), rnn_test_acc])
summary_rows.append(["LSTM", final_train_acc(history_lstm), best_val_acc(history_lstm), lstm_test_acc])
summary_rows.append(["GRU", final_train_acc(history_gru), best_val_acc(history_gru), gru_test_acc])
summary_rows.append(["Stacked LSTM + Dropout", final_train_acc(history_stacked_lstm), best_val_acc(history_stacked_lstm), stacked_lstm_test_acc])
summary_rows.append(["Encoder-Decoder LSTM", final_train_acc(history_encdec), best_val_acc(history_encdec), encdec_test_acc])
summary_rows.append(["LSTM + GRU Hybrid", final_train_acc(history_hybrid), best_val_acc(history_hybrid), hybrid_test_acc])

# TODO 5: Create DataFrame
results_df = pd.DataFrame(
    summary_rows,
    columns=["Model", "Final Train Acc", "Best Val Acc", "Test Acc"]
)

# TODO 6: Sort by test accuracy
results_df = results_df.sort_values(by="Test Acc", ascending=False).reset_index(drop=True)

# Display results
results_df

Test (Vanilla RNN): loss=0.6938, acc=0.5028
Test (LSTM): loss=0.5117, acc=0.7980
Test (GRU): loss=0.5592, acc=0.7901
Test (Stacked LSTM + Dropout): loss=0.5297, acc=0.7650
Test (Encoder-Decoder LSTM): loss=0.4349, acc=0.8549
Test (LSTM + GRU Hybrid): loss=0.4648, acc=0.8434


,Model,Final Train Acc,Best Val Acc,Test Acc
0,Encoder-Decoder LSTM,0.98180,0.8672,0.85488
1,LSTM + GRU Hybrid,0.93625,0.8676,0.84344
2,LSTM,0.83410,0.8096,0.79804
3,GRU,0.86290,0.7994,0.79012
4,Stacked LSTM + Dropout,0.72870,0.7748,0.76500
5,Vanilla RNN,0.72340,0.5114,0.50280


---

# Results and Discussions

Use the results above to answer the discussion questions below. You may revise the answers based on your actual experimental results (**Q1-Q11**).

---

## **Q12** — Which model performed best overall, and why might it outperform the others?  

**Answer:** ...

---
Based on my experimental results, the Encoder-Decoder LSTM performed the best overall, achieving the highest test accuracy (85.49%) and the lowest test loss (0.4349). It likely outperformed the simpler models because the encoder forces the network to compress the entire 250-token sequence into a single, dense context vector before passing it to the decoder. This architecture acts as an information bottleneck, filtering out local sequential noise and forcing the network to learn a robust, big-picture summary of the review's overall sentiment. The LSTM + GRU Hybrid was a close second (84.34%), showing that combining different gating mechanisms is also highly effective.

## **Q13** — Why does a vanilla RNN usually struggle more on long reviews?  

**Answer:** ...

---
The Vanilla RNN performed very poorly (50.28% test accuracy) because it suffers severely from the vanishing gradient problem. During backpropagation over long sequences (like our 250-token reviews), gradients are repeatedly multiplied by the network's weight matrix. These gradients exponentially shrink toward zero, meaning the RNN stops updating its weights with respect to the words at the beginning of the review. It essentially "forgets" the early context and cannot maintain the long-term dependencies required to accurately gauge the overall sentiment.

## **Q14** — Why might a more complex model not always outperform a simpler GRU or LSTM?  

**Answer:** ...

---
While complex models won in this specific test, my data shows they are highly prone to overfitting. For example, the Encoder-Decoder LSTM reached a massive Final Training Accuracy of 98.18%, but its Test Accuracy was only 85.49%. Complex architectures contain significantly more parameters, giving them a massive capacity to memorize the training data. If a dataset isn't complex enough to warrant that depth, or if aggressive regularization (like dropout) isn't used, deeper models will overfit early in training. Furthermore, the Stacked LSTM actually performed worse (76.50% test acc) than the single-layer LSTM (79.80%), proving that simply adding more layers without careful tuning can make the network harder to optimize.

## **Q15** — What is the main idea behind the encoder–decoder classifier used here?  

**Answer:** ...

---
The primary idea is to create an explicit structural bottleneck. Instead of making a classification decision based directly on the final hidden state of a single pass, the Encoder reads the sequence and is forced to compress its entire meaning into a single, global "context vector." The Decoder then takes this condensed representation and uses it to generate a refined hidden state, which is passed to the dense classifier.

### 🎉 Congratulations!

You have successfully completed **P5 — Sequence Modeling with RNNs**. Excellent work applying and comparing **RNN, LSTM, and GRU architectures** for a **sequence-based text classification** task using the **IMDB Movie Reviews dataset**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q11)** have been executed successfully
- All **Markdown responses (Q12–Q15)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.